# 16 – Vertiefung: Morse-Code (Dictionary + Tonausgabe)

Dieses Notebook nutzt ein **Dictionary** als Lookup-Tabelle: Zeichen → Morse-Code. Du lernst, Text in Morse zu übersetzen und die Folge als **Ton** im Notebook abzuspielen (über `IPython.display.Audio`).

**Navigation:** [← 16_Dictionaries (Hauptkapitel)](16_Dictionaries.ipynb) | Morse-Code & Ton

## 1. Dictionary: Zeichen → Morse-Code

Jedes Zeichen (Buchstabe, Ziffer) wird auf eine Morse-Folge aus Punkten (`.`) und Strichen (`-`) abgebildet. Ein **Dictionary** ist dafür ideal: Schlüssel = Zeichen, Wert = Morse-String.

In [4]:
# Morse-Code Lookup-Tabelle (A–Z, 0–9)
ZEICHEN_ZU_MORSE = {
    "A": ".-", "B": "-...", "C": "-.-.", "D": "-..", "E": ".", "F": "..-.",
    "G": "--.", "H": "....", "I": "..", "J": ".---", "K": "-.-", "L": ".-..",
    "M": "--", "N": "-.", "O": "---", "P": ".--.", "Q": "--.-", "R": ".-.",
    "S": "...", "T": "-", "U": "..-", "V": "...-", "W": ".--", "X": "-..-",
    "Y": "-.--", "Z": "--..",
    "0": "-----", "1": ".----", "2": "..---", "3": "...--", "4": "....-",
    "5": ".....", "6": "-....", "7": "--...", "8": "---..", "9": "----.",
    " ": " "  # Leerzeichen = Pause zwischen Wörtern
}

text = "CQ CQ DE oe5chm"
for zeichen in text.upper():
    morse = ZEICHEN_ZU_MORSE.get(zeichen, "")
    print(f"{zeichen!r} -> {morse!r}")

'C' -> '-.-.'
'Q' -> '--.-'
' ' -> ' '
'C' -> '-.-.'
'Q' -> '--.-'
' ' -> ' '
'D' -> '-..'
'E' -> '.'
' ' -> ' '
'O' -> '---'
'E' -> '.'
'5' -> '.....'
'C' -> '-.-.'
'H' -> '....'
'M' -> '--'


## 2. Text in Morse-String umwandeln

Mit einer Schleife über alle Zeichen und **`.get(zeichen, "")`** holst du die Morse-Folge. Unbekannte Zeichen (z. B. Umlaute) liefern den Default `""` und werden übersprungen.

In [5]:
def text_zu_morse(text, zeichen_zu_morse):
    """Wandelt Text in eine Morse-Zeichenkette um (Punkt . Strich - Leerzeichen zwischen Buchstaben)."""
    zeichenkette = []
    for zeichen in text.upper():
        m = zeichen_zu_morse.get(zeichen, "")
        if m == " ":
            zeichenkette.append(" ")  # Wortabstand
        elif m:
            zeichenkette.append(m)
    return " ".join(zeichenkette)  # zwischen Buchstaben: Leerzeichen

beispiel = "SOS"
print(text_zu_morse(beispiel, ZEICHEN_ZU_MORSE))
print(text_zu_morse("Hallo", ZEICHEN_ZU_MORSE))

... --- ...
.... .- .-.. .-.. ---


## 3. Ton im Notebook ausgeben

**IPython.display.Audio** spielt ein 1D-NumPy-Array als Ton ab (Abtastrate angeben). Wir bauen einen großen Waveform-Array:
- **Punkt (dit):** kurzer Sinus-Ton (1 Einheit)
- **Strich (dah):** langer Sinus-Ton (3 Einheiten)
- **Pausen:** Stille (1 Einheit zwischen Symbolen, 3 zwischen Buchstaben, 7 zwischen Wörtern)

Standard-Timing: 1 Einheit ≈ 0,05 s (schnell) bis 0,1 s (langsam).

In [ ]:
import numpy as np
import IPython.display as ipd

SAMPLE_RATE = 22050   # Hz
FREQUENZ = 600       # Hz (typisch für Morse)
EINHEIT_SEK = 0.08   # Dauer einer Einheit in Sekunden (Punkt = 1, Strich = 3)
AMPLITUDE = 0.3

def ton_segment(dauer_sec, ton_an=True):
    """Erzeugt ein Waveform-Segment: Sinus oder Stille."""
    n = int(SAMPLE_RATE * dauer_sec)
    t = np.linspace(0, dauer_sec, n, endpoint=False)
    if ton_an:
        return AMPLITUDE * np.sin(2 * np.pi * FREQUENZ * t)
    return np.zeros(n)

def morse_zu_ton(morse_string):
    """Wandelt Morse-String (z. B. '. - . . .') in ein Audio-Array um."""
    segmente = []
    einheit = EINHEIT_SEK
    letztes_war_leer = False
    for zeichen in morse_string:
        if zeichen == ".":
            letztes_war_leer = False
            segmente.append(ton_segment(einheit, True))
            segmente.append(ton_segment(einheit, False))
        elif zeichen == "-":
            letztes_war_leer = False
            segmente.append(ton_segment(3 * einheit, True))
            segmente.append(ton_segment(einheit, False))
        elif zeichen == " ":
            # 3 Einheiten zwischen Buchstaben; bei Wortabstand (doppeltes Leerzeichen) 4 weitere
            pause = (4 * einheit) if letztes_war_leer else (3 * einheit)
            letztes_war_leer = True
            segmente.append(ton_segment(pause, False))
    if segmente:
        return np.concatenate(segmente)
    return np.array([0.0])

def spiele_morse(text):
    """Text in Morse umwandeln und als Ton abspielen."""
    morse = text_zu_morse(text, ZEICHEN_ZU_MORSE)
    # Doppeltes Leerzeichen = Wortabstand (7 Einheiten); in morse_zu_ton wird 1 Leerzeichen = 3, zweites = +4
    waveform = morse_zu_ton(morse)
    print(f"Morse: {morse}")
    return ipd.Audio(waveform, rate=SAMPLE_RATE)

# Abspielen: Klick auf Play im ausgegebenen Audio-Widget
spiele_morse("CQ CQ DE oe5chm")

Morse: -.-. --.-   -.-. --.-   -.. .   --- . ..... -.-. .... --


## 4. Übungsideen

- **Erweiterung:** Umlaute (Ä, Ö, Ü) im Dictionary ergänzen (z. B. wie A/O/U mit Länge 2).
- **Rückrichtung:** Ein Dictionary `morse_zu_zeichen` bauen (z. B. aus `items()` tauschen) und Morse-String wieder in Text decodieren.
- **Geschwindigkeit:** `EINHEIT_SEK` verkleinern (schneller) oder vergrößern (langsamer).